In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:openai/gpt-oss-120b")

def create_story_chain():
    # Create a story generation chain that takes a topic as input and generates a short story.
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that creates short stories based on user input."),
        ("user", "Write a short story about a {topic}.")
    ])
    
    # Create a story analysis chain that takes a story as input and classifies it into a genre.
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that analyzes short stories."),
        ("user", "Analyze the following story and classify it into one of the following genres: fantasy, science fiction, mystery, or romance.\n\n{story}")
    ])

    story_chain = (
        story_prompt | llm | StrOutputParser()
    )

    def analyze_story(story):
        return {"story": story}

    analysis_chain = (
        story_chain | RunnableLambda(analyze_story) | analysis_prompt | llm | StrOutputParser()
    )
    return analysis_chain



In [ ]:
response = create_story_chain()
result = response.invoke("a dragon and a knight")
result


'**Genre Classification: Fantasy**\n\n**Why it fits the fantasy genre**\n\n| Element | Explanation |\n|---------|-------------|\n| **Setting** | A medieval‑style kingdom (Valtara), mountains, caverns, and a royal court – classic fantasy world‑building. |\n| **Supernatural Creatures** | A copper‑scaled dragon, a dragon hatchling, and the notion of “ancient magic” are hallmarks of fantasy. |\n| **Magic & Mythic Powers** | The dragon’s breath is golden vapor, a “spark of ancient magic” passes between dragon and knight, and the cavern’s mineral veins pulse with life. |\n| **Heroic Quest** | Sir Alden’s journey to confront (and ultimately befriend) the dragon follows the traditional fantasy quest narrative. |\n| **Themes** | Themes of honor, pact‑making, and the bridging of myth and humanity are common in fantasy literature. |\n| **Absence of Sci‑Fi/Mystery/Romance Elements** | No advanced technology or scientific explanation (science‑fiction), no investigative plot or whodunit (mystery), a